# Spark SQL 工具包

本笔记本展示了如何使用代理与 `Spark SQL` 进行交互。与 [SQL 数据库代理](/docs/integrations/tools/sql_database) 类似，它旨在解决关于 `Spark SQL` 的一般性查询并促进错误恢复。

**注意：请注意，由于此代理目前处于积极开发阶段，并非所有答案都可能正确。此外，不能保证在某些问题下代理不会在您的 Spark 集群上执行 DML 语句。在处理敏感数据时请谨慎使用！**

## 初始化

In [1]:
from langchain_community.agent_toolkits import SparkSQLToolkit, create_spark_sql_agent
from langchain_community.utilities.spark_sql import SparkSQL
from langchain_openai import ChatOpenAI

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()
schema = "langchain_example"
spark.sql(f"CREATE DATABASE IF NOT EXISTS {schema}")
spark.sql(f"USE {schema}")
csv_file_path = "titanic.csv"
table = "titanic"
spark.read.csv(csv_file_path, header=True, inferSchema=True).write.saveAsTable(table)
spark.table(table).show()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
23/05/18 16:03:10 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|          Ticket|   Fare|Cabin|Embarked|
+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
|          1|       0|     3|Braund, Mr. Owen ...|  male|22.0|    1|    0|       A/5 21171|   7.25| null|       S|
|          2|       1|     1|Cumings, Mrs. Joh...|female|38.0|    1|    0|        PC 17599|71.2833|  C85|       C|
|          3|       1|     3|Heikkinen, Miss. ...|female|26.0|    0|    0|STON/O2. 3101282|  7.925| null|       S|
|          4|       1|     1|Futrelle, Mrs. Ja...|female|35.0|    1|    0|          113803|   53.1| C123|       S|
|          5|       0|     3|Allen, Mr. Willia...|  male|35.0|    0|    0|          373450|   8.05| null|       S|
|          6|       0|     3|    Moran, Mr. James|  male|null|    0|    0|      

In [3]:
# Note, you can also connect to Spark via Spark connect. For example:
# db = SparkSQL.from_uri("sc://localhost:15002", schema=schema)
spark_sql = SparkSQL(schema=schema)
llm = ChatOpenAI(temperature=0)
toolkit = SparkSQLToolkit(db=spark_sql, llm=llm)
agent_executor = create_spark_sql_agent(llm=llm, toolkit=toolkit, verbose=True)

## 示例：描述表格

This is a table that shows the number of users for each plan.

这是一个显示每个计划用户数量的表格。

```md
| Plan | Users |
|---|---|
| Free | 1000 |
| Pro | 500 |
| Enterprise | 100 |

In [4]:
agent_executor.run("Describe the titanic table")



> Entering new AgentExecutor chain...
Action: list_tables_sql_db
Action Input: 
Observation: titanic
Thought:I found the titanic table. Now I need to get the schema and sample rows for the titanic table.
Action: schema_sql_db
Action Input: titanic
Observation: CREATE TABLE langchain_example.titanic (
  PassengerId INT,
  Survived INT,
  Pclass INT,
  Name STRING,
  Sex STRING,
  Age DOUBLE,
  SibSp INT,
  Parch INT,
  Ticket STRING,
  Fare DOUBLE,
  Cabin STRING,
  Embarked STRING)
;

/*
3 rows from titanic table:
PassengerId	Survived	Pclass	Name	Sex	Age	SibSp	Parch	Ticket	Fare	Cabin	Embarked
1	0	3	Braund, Mr. Owen Harris	male	22.0	1	0	A/5 21171	7.25	None	S
2	1	1	Cumings, Mrs. John Bradley (Florence Briggs Thayer)	female	38.0	1	0	PC 17599	71.2833	C85	C
3	1	3	Heikkinen, Miss. Laina	female	26.0	0	0	STON/O2. 3101282	7.925	None	S
*/
Thought:I now know the schema and sample rows for the titanic table.
Final Answer: The titanic table has the following columns: PassengerId (INT), Survived (

'The titanic table has the following columns: PassengerId (INT), Survived (INT), Pclass (INT), Name (STRING), Sex (STRING), Age (DOUBLE), SibSp (INT), Parch (INT), Ticket (STRING), Fare (DOUBLE), Cabin (STRING), and Embarked (STRING). Here are some sample rows from the table: \n\n1. PassengerId: 1, Survived: 0, Pclass: 3, Name: Braund, Mr. Owen Harris, Sex: male, Age: 22.0, SibSp: 1, Parch: 0, Ticket: A/5 21171, Fare: 7.25, Cabin: None, Embarked: S\n2. PassengerId: 2, Survived: 1, Pclass: 1, Name: Cumings, Mrs. John Bradley (Florence Briggs Thayer), Sex: female, Age: 38.0, SibSp: 1, Parch: 0, Ticket: PC 17599, Fare: 71.2833, Cabin: C85, Embarked: C\n3. PassengerId: 3, Survived: 1, Pclass: 3, Name: Heikkinen, Miss. Laina, Sex: female, Age: 26.0, SibSp: 0, Parch: 0, Ticket: STON/O2. 3101282, Fare: 7.925, Cabin: None, Embarked: S'

## 示例：运行查询

This section provides examples of how to run queries against your data.

本节将提供针对您的数据运行查询的示例。

```js
// Import the necessary libraries
import { gql, ApolloClient, InMemoryCache } from '@apollo/client';

// Create an Apollo Client instance
const client = new ApolloClient({
  uri: 'https://countries.trevorblades.com/',
  cache: new InMemoryCache(),
});

// Define the GraphQL query
const GET_CONTINENTS = gql`
  query GetContinents {
    continents {
      code
      name
    }
  }
`;

// Run the query
client
  .query({
    query: GET_CONTINENTS,
  })
  .then((result) => console.log(result));
```

```js
// 导入必要的库
import { gql, ApolloClient, InMemoryCache } from '@apollo/client';

// 创建一个 Apollo Client 实例
const client = new ApolloClient({
  uri: 'https://countries.trevorblades.com/',
  cache: new InMemoryCache(),
});

// 定义 GraphQL 查询
const GET_CONTINENTS = gql`
  query GetContinents {
    continents {
      code
      name
    }
  }
`;

// 运行查询
client
  .query({
    query: GET_CONTINENTS,
  })
  .then((result) => console.log(result));
```

This example demonstrates how to fetch a list of continents and their codes from the Countries API.

此示例演示了如何从 Countries API 获取大陆列表及其代码。

### Fetching data with variables

### 使用变量获取数据

You can also pass variables to your queries to make them more dynamic.

您还可以将变量传递给查询，使它们更具动态性。

```js
// Define a query that accepts a variable
const GET_COUNTRY_BY_CODE = gql`
  query GetCountry($code: ID!) {
    country(code: $code) {
      name
      native
      phone
      capital
      emoji
      currency
      languages {
        code
        name
      }
    }
  }
`;

// Define the variables to pass to the query
const countryCode = 'CA'; // Example: Canada

// Run the query with variables
client
  .query({
    query: GET_COUNTRY_BY_CODE,
    variables: { code: countryCode },
  })
  .then((result) => console.log(result));
```

```js
// 定义一个接受变量的查询
const GET_COUNTRY_BY_CODE = gql`
  query GetCountry($code: ID!) {
    country(code: $code) {
      name
      native
      phone
      capital
      emoji
      currency
      languages {
        code
        name
      }
    }
  }
`;

// 定义要传递给查询的变量
const countryCode = 'CA'; // 示例：加拿大

// 使用变量运行查询
client
  .query({
    query: GET_COUNTRY_BY_CODE,
    variables: { code: countryCode },
  })
  .then((result) => console.log(result));
```

This example fetches details for a specific country by providing its code as a variable.

此示例通过提供国家代码作为变量来获取特定国家的详细信息。

### Mutations

### Mutations

Mutations are used to modify data on the server. Here's an example of how to perform a mutation:

Mutations 用于修改服务器上的数据。以下是如何执行 mutation 的示例：

```js
// Define a mutation
const ADD_TODO = gql`
  mutation AddTodo($text: String!) {
    addTodo(text: $text) {
      id
      text
      completed
    }
  }
`;

// Define the variables for the mutation
const todoText = 'Learn GraphQL';

// Run the mutation
client
  .mutate({
    mutation: ADD_TODO,
    variables: { text: todoText },
  })
  .then((result) => console.log(result));
```

```js
// 定义一个 mutation
const ADD_TODO = gql`
  mutation AddTodo($text: String!) {
    addTodo(text: $text) {
      id
      text
      completed
    }
  }
`;

// 定义 mutation 的变量
const todoText = 'Learn GraphQL';

// 运行 mutation
client
  .mutate({
    mutation: ADD_TODO,
    variables: { text: todoText },
  })
  .then((result) => console.log(result));
```

This example adds a new todo item to a list.

此示例向列表中添加一个新的待办事项。

In [5]:
agent_executor.run("whats the square root of the average age?")



> Entering new AgentExecutor chain...
Action: list_tables_sql_db
Action Input: 
Observation: titanic
Thought:I should check the schema of the titanic table to see if there is an age column.
Action: schema_sql_db
Action Input: titanic
Observation: CREATE TABLE langchain_example.titanic (
  PassengerId INT,
  Survived INT,
  Pclass INT,
  Name STRING,
  Sex STRING,
  Age DOUBLE,
  SibSp INT,
  Parch INT,
  Ticket STRING,
  Fare DOUBLE,
  Cabin STRING,
  Embarked STRING)
;

/*
3 rows from titanic table:
PassengerId	Survived	Pclass	Name	Sex	Age	SibSp	Parch	Ticket	Fare	Cabin	Embarked
1	0	3	Braund, Mr. Owen Harris	male	22.0	1	0	A/5 21171	7.25	None	S
2	1	1	Cumings, Mrs. John Bradley (Florence Briggs Thayer)	female	38.0	1	0	PC 17599	71.2833	C85	C
3	1	3	Heikkinen, Miss. Laina	female	26.0	0	0	STON/O2. 3101282	7.925	None	S
*/
Thought:There is an Age column in the titanic table. I should write a query to calculate the average age and then find the square root of the result.
Action: query_checker

'The square root of the average age is approximately 5.45.'

In [6]:
agent_executor.run("What's the name of the oldest survived passenger?")



> Entering new AgentExecutor chain...
Action: list_tables_sql_db
Action Input: 
Observation: titanic
Thought:I should check the schema of the titanic table to see what columns are available.
Action: schema_sql_db
Action Input: titanic
Observation: CREATE TABLE langchain_example.titanic (
  PassengerId INT,
  Survived INT,
  Pclass INT,
  Name STRING,
  Sex STRING,
  Age DOUBLE,
  SibSp INT,
  Parch INT,
  Ticket STRING,
  Fare DOUBLE,
  Cabin STRING,
  Embarked STRING)
;

/*
3 rows from titanic table:
PassengerId	Survived	Pclass	Name	Sex	Age	SibSp	Parch	Ticket	Fare	Cabin	Embarked
1	0	3	Braund, Mr. Owen Harris	male	22.0	1	0	A/5 21171	7.25	None	S
2	1	1	Cumings, Mrs. John Bradley (Florence Briggs Thayer)	female	38.0	1	0	PC 17599	71.2833	C85	C
3	1	3	Heikkinen, Miss. Laina	female	26.0	0	0	STON/O2. 3101282	7.925	None	S
*/
Thought:I can use the titanic table to find the oldest survived passenger. I will query the Name and Age columns, filtering by Survived and ordering by Age in descending 

'The oldest survived passenger is Barkworth, Mr. Algernon Henry Wilson, who was 80 years old.'